In [4]:
from collections import namedtuple
import random

# ============================================================
# ใช้เก็บข้อมูลสถานะปัจจุบันของเกม
# ============================================================
State = namedtuple(
    'State',
    ['turn', 'score', 'pieces', 'legal_moves', 'finished']
)

INF = float('inf')


# ============================================================
# คลาสพื้นฐานสำหรับเกมแบบค้นหาสถานะ
# ============================================================
class GameBase:

    def actions(self, state):
        raise NotImplementedError

    def result(self, state, move):
        raise NotImplementedError

    def utility(self, state, player):
        raise NotImplementedError

    def terminal_test(self, state):
        return not self.actions(state)

    def to_move(self, state):
        return state.turn

    def display(self, state):
        print(state)


# ============================================================
# เกม Hexapawn
# ============================================================
class Hexapawn(GameBase):

    def __init__(self, rows=3, cols=3):
        self.rows = rows
        self.cols = cols

        pieces = {}

        # วางตัวหมากฝ่ายดำไว้ด้านบน
        for col in range(cols):
            pieces[(col, 0)] = 'B'

        # วางตัวหมากฝ่ายขาวไว้ด้านล่าง
        for col in range(cols):
            pieces[(col, rows - 1)] = 'W'

        first_moves = self.find_moves(pieces, 'W')

        self.initial = State(
            turn='W',
            score=0,
            pieces=pieces,
            legal_moves=first_moves,
            finished=False
        )

    def find_moves(self, pieces, player):

        possible = []

        step = -1 if player == 'W' else 1
        enemy = 'B' if player == 'W' else 'W'

        for (col, row), piece in pieces.items():

            if piece != player:
                continue

            next_row = row + step

            if not (0 <= next_row < self.rows):
                continue

            # เดินตรงได้เมื่อช่องข้างหน้าว่าง
            if (col, next_row) not in pieces:
                possible.append(
                    ((col, row), (col, next_row))
                )

            # กินหมากฝ่ายตรงข้ามในแนวทแยง
            for offset in (-1, 1):

                next_col = col + offset

                if (
                    0 <= next_col < self.cols
                    and pieces.get((next_col, next_row)) == enemy
                ):
                    possible.append(
                        ((col, row), (next_col, next_row))
                    )

        return possible

    def actions(self, state):
        return state.legal_moves

    def result(self, state, move):

        start, end = move
        player = state.turn

        pieces = dict(state.pieces)

        # ย้ายตัวหมากออกจากตำแหน่งเก่า
        del pieces[start]

        # วางไว้ที่ตำแหน่งใหม่
        pieces[end] = player

        enemy = 'B' if player == 'W' else 'W'

        enemy_moves = self.find_moves(
            pieces,
            enemy
        )

        finished, score = self.check_result(
            pieces,
            end,
            player,
            enemy_moves
        )

        return State(
            turn=enemy,
            score=score,
            pieces=pieces,
            legal_moves=enemy_moves,
            finished=finished
        )

    def check_result(
        self,
        pieces,
        destination,
        player,
        enemy_moves
    ):

        col, row = destination
        enemy = 'B' if player == 'W' else 'W'

        # ฝ่ายขาวไปถึงด้านบน
        if player == 'W' and row == 0:
            return True, 1

        # ฝ่ายดำไปถึงด้านล่าง
        if player == 'B' and row == self.rows - 1:
            return True, -1

        # ฝ่ายตรงข้ามไม่มีหมากเหลือ
        if not any(
            piece == enemy
            for piece in pieces.values()
        ):
            return True, 0

        # ฝ่ายตรงข้ามไม่สามารถเดินต่อได้
        if not enemy_moves:
            return True, 0

        return False, 0

    def utility(self, state, player):

        if player == 'W':
            return state.score

        return -state.score

    def terminal_test(self, state):
        return state.finished

    def display(self, state):

        print(
            '    ' +
            '   '.join(
                str(i)
                for i in range(self.cols)
            )
        )

        print('  +' + '---+' * self.cols)

        for row in range(self.rows):

            line = []

            for col in range(self.cols):
                line.append(
                    state.pieces.get(
                        (col, row),
                        '.'
                    )
                )

            print(
                f'{row} | ' +
                ' | '.join(line) +
                ' |'
            )

            print(
                '  +' +
                '---+' * self.cols
            )

        print()


# ============================================================
# ค้นหาตาเดินที่ดีที่สุดด้วย Alpha-Beta Pruning
# ============================================================
def alpha_beta_search(state, game):

    root_player = game.to_move(state)

    def maximize(state, alpha, beta):

        if game.terminal_test(state):
            return game.utility(
                state,
                root_player
            )

        value = -INF

        for move in game.actions(state):

            next_state = game.result(
                state,
                move
            )

            value = max(
                value,
                minimize(
                    next_state,
                    alpha,
                    beta
                )
            )

            if value >= beta:
                return value

            alpha = max(
                alpha,
                value
            )

        return value

    def minimize(state, alpha, beta):

        if game.terminal_test(state):
            return game.utility(
                state,
                root_player
            )

        value = INF

        for move in game.actions(state):

            next_state = game.result(
                state,
                move
            )

            value = min(
                value,
                maximize(
                    next_state,
                    alpha,
                    beta
                )
            )

            if value <= alpha:
                return value

            beta = min(
                beta,
                value
            )

        return value

    best_value = -INF
    selected_move = None

    for move in game.actions(state):

        value = minimize(
            game.result(state, move),
            best_value,
            INF
        )

        if value > best_value:
            best_value = value
            selected_move = move

    return selected_move


# ============================================================
# ตัวเล่นของ AI
# ============================================================
def ai_player(game, state):

    move = alpha_beta_search(
        state,
        game
    )

    print(
        f'AI เลือกเดินจาก '
        f'{move[0]} ไป {move[1]}'
    )

    return move


# ============================================================
# ตัวเล่นแบบสุ่ม
# ============================================================
def random_player(game, state):

    return random.choice(
        game.actions(state)
    )


# ============================================================
# รับตาเดินจากผู้เล่น
# ============================================================
def human_player(game, state):

    moves = game.actions(state)

    print('ตัวเลือกสำหรับตานี้:')

    for index, move in enumerate(moves):

        print(
            f'{index}: '
            f'{move[0]} → {move[1]}'
        )

    while True:

        try:

            choice = int(
                input(
                    'กรอกหมายเลขที่ต้องการเดิน: '
                )
            )

            if 0 <= choice < len(moves):
                return moves[choice]

            print('หมายเลขที่เลือกไม่ถูกต้อง')

        except ValueError:
            print('กรุณากรอกเป็นตัวเลขเท่านั้น')


# ============================================================
# ควบคุมการเล่นตั้งแต่เริ่มจนจบเกม
# ============================================================
def start_game(game, white, black):

    state = game.initial

    players = {
        'W': white,
        'B': black
    }

    round_no = 1

    while True:

        print(
            f'\n---------- รอบที่ {round_no} ----------'
        )

        game.display(state)

        if game.terminal_test(state):

            if state.score == 1:
                print('White เป็นฝ่ายชนะ')

            elif state.score == -1:
                print('Black เป็นฝ่ายชนะ')

            else:
                print('เกมจบแบบเสมอ')

            return state.score

        current = game.to_move(state)

        if current == 'W':
            print('ถึงตาของผู้เล่นฝ่าย White')
        else:
            print('ถึงตาของ AI ฝ่าย Black')

        move = players[current](
            game,
            state
        )

        state = game.result(
            state,
            move
        )

        round_no += 1


# ============================================================
# เริ่มต้นโปรแกรม
# ============================================================
print('==============================')
print('        HEXAPAWN GAME')
print('==============================')
print('ผู้เล่นรับบทเป็น White')
print('คอมพิวเตอร์รับบทเป็น Black')
print()
print('วิธีเล่น')
print('1. หมากสามารถเดินตรงได้เมื่อช่องด้านหน้าว่าง')
print('2. การกินหมากต้องเดินในแนวทแยง')
print('3. ผู้ที่พาหมากไปถึงฝั่งตรงข้ามได้ก่อนถือว่าชนะ')
print('4. หากเกมหยุดด้วยเงื่อนไขอื่นจะถือว่าเสมอ')
print()

game = Hexapawn(
    rows=3,
    cols=3
)

start_game(
    game,
    human_player,
    ai_player
)

        HEXAPAWN GAME
ผู้เล่นรับบทเป็น White
คอมพิวเตอร์รับบทเป็น Black

วิธีเล่น
1. หมากสามารถเดินตรงได้เมื่อช่องด้านหน้าว่าง
2. การกินหมากต้องเดินในแนวทแยง
3. ผู้ที่พาหมากไปถึงฝั่งตรงข้ามได้ก่อนถือว่าชนะ
4. หากเกมหยุดด้วยเงื่อนไขอื่นจะถือว่าเสมอ


---------- รอบที่ 1 ----------
    0   1   2
  +---+---+---+
0 | B | B | B |
  +---+---+---+
1 | . | . | . |
  +---+---+---+
2 | W | W | W |
  +---+---+---+

ถึงตาของผู้เล่นฝ่าย White
ตัวเลือกสำหรับตานี้:
0: (0, 2) → (0, 1)
1: (1, 2) → (1, 1)
2: (2, 2) → (2, 1)
กรอกหมายเลขที่ต้องการเดิน: 0

---------- รอบที่ 2 ----------
    0   1   2
  +---+---+---+
0 | B | B | B |
  +---+---+---+
1 | W | . | . |
  +---+---+---+
2 | . | W | W |
  +---+---+---+

ถึงตาของ AI ฝ่าย Black
AI เลือกเดินจาก (1, 0) ไป (1, 1)

---------- รอบที่ 3 ----------
    0   1   2
  +---+---+---+
0 | B | . | B |
  +---+---+---+
1 | W | B | . |
  +---+---+---+
2 | . | W | W |
  +---+---+---+

ถึงตาของผู้เล่นฝ่าย White
ตัวเลือกสำหรับตานี้:
0: (2, 2) → (2, 1)
1: (2, 2) → (1, 1)


0